In [1]:
# Manipulation des tables Parquet et Arrow.
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import fs

# Fonction scikit-learn utilisée pour diviser les données.
from sklearn.model_selection import train_test_split

In [2]:
# Crée l'objet PyArrow qui communique avec Amazon S3.
s3 = fs.S3FileSystem(region="us-east-1")

# Nom du bucket S3.
BUCKET_NAME = "mushroom-ml-mikael-2026-000"

# Fichier source contenant les données nettoyées et les ID.
SOURCE_PATH = (
    f"{BUCKET_NAME}/raw/parquet/"
    "mushrooms_CleanedNamesID.parquet"
)

# Dossier de destination pour les trois datasets.
OUTPUT_DIR = f"{BUCKET_NAME}/processed/parquet"

# Chemins complets des fichiers de sortie.
TRAIN_PATH = f"{OUTPUT_DIR}/train.parquet"
VALIDATION_PATH = f"{OUTPUT_DIR}/validation.parquet"
TEST_PATH = f"{OUTPUT_DIR}/test.parquet"

In [3]:
# Lit le fichier Parquet directement depuis S3.
table = pq.read_table(
    SOURCE_PATH,
    filesystem=s3
)

# Affiche quelques informations pour vérifier le chargement.
print("Nombre de lignes :", table.num_rows)
print("Colonnes :", table.column_names)

Nombre de lignes : 2436
Colonnes : ['image', 'label', 'bbox', 'split', 'mushroom_name', 'edibility', 'entry_id']


In [4]:
# Convertit temporairement la table PyArrow en DataFrame Pandas.
dataframe = table.to_pandas()

# Affiche les cinq premières lignes.
display(dataframe.head())

,image,label,bbox,split,mushroom_name,edibility,entry_id
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[748.0, 285.0, 1011.75, 505.0]",test,Amanita phalloides,False,000000
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[178.0, 113.0, 445.5, 548.0]",test,Amanita phalloides,False,000001
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[471.0, 368.0, 594.75, 571.75]",test,Amanita phalloides,False,000002
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[553.0, 20.0, 854.25, 315.0]",test,Amanita phalloides,False,000003
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Amanita_phalloides_poisonous,"[484.0, 138.0, 672.75, 361.75]",test,Amanita phalloides,False,000004


In [5]:
# Sépare 70 % des données pour le train.
#
# Les 30 % restants seront ensuite partagés entre
# validation et test.
train_df, remaining_df = train_test_split(
    dataframe,
    test_size=0.30,
    random_state=42,
    stratify=dataframe["mushroom_name"]
)

In [6]:
# Divise les 30 % restants en deux parties égales.
#
# Cela donne :
# - 15 % validation
# - 15 % test
validation_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    random_state=42,
    stratify=remaining_df["mushroom_name"]
)

# Calcule la taille totale du dataset.
total_rows = len(dataframe)

# Affiche les proportions réelles obtenues.
print(f"Train : {len(train_df) / total_rows:.2%}")
print(f"Validation : {len(validation_df) / total_rows:.2%}")
print(f"Test : {len(test_df) / total_rows:.2%}")

Train : 69.99%
Validation : 14.98%
Test : 15.02%


In [7]:
# Affiche la proportion de chaque espèce dans le dataset complet.
print("Dataset complet")
display(
    dataframe["mushroom_name"]
    .value_counts(normalize=True)
    .sort_index()
)

# Affiche la proportion de chaque espèce dans le train.
print("Train")
display(
    train_df["mushroom_name"]
    .value_counts(normalize=True)
    .sort_index()
)

# Affiche la proportion de chaque espèce dans la validation.
print("Validation")
display(
    validation_df["mushroom_name"]
    .value_counts(normalize=True)
    .sort_index()
)

# Affiche la proportion de chaque espèce dans le test.
print("Test")
display(
    test_df["mushroom_name"]
    .value_counts(normalize=True)
    .sort_index()
)

Dataset complet


mushroom_name
Agaricus bisporus            0.098112
Amanita phalloides           0.057471
Armillaria mellea            0.086207
Boletus edulis               0.087438
Cantharellus cibarius        0.035714
Clitocybe dealbata           0.073481
Entoloma sinuatum            0.119869
Hygrophoropsis aurantiaca    0.101396
Hypholoma fasciculare        0.076765
Lactarius resimus            0.070197
Pleurotus ostreatus          0.076765
Rubroboletus satanas         0.116585
Name: proportion, dtype: float64

Train


mushroom_name
Agaricus bisporus            0.097947
Amanita phalloides           0.057478
Armillaria mellea            0.086217
Boletus edulis               0.087390
Cantharellus cibarius        0.035777
Clitocybe dealbata           0.073314
Entoloma sinuatum            0.119648
Hygrophoropsis aurantiaca    0.101466
Hypholoma fasciculare        0.076833
Lactarius resimus            0.070381
Pleurotus ostreatus          0.076833
Rubroboletus satanas         0.116716
Name: proportion, dtype: float64

Validation


mushroom_name
Agaricus bisporus            0.098630
Amanita phalloides           0.057534
Armillaria mellea            0.084932
Boletus edulis               0.087671
Cantharellus cibarius        0.035616
Clitocybe dealbata           0.073973
Entoloma sinuatum            0.120548
Hygrophoropsis aurantiaca    0.101370
Hypholoma fasciculare        0.076712
Lactarius resimus            0.071233
Pleurotus ostreatus          0.076712
Rubroboletus satanas         0.115068
Name: proportion, dtype: float64

Test


mushroom_name
Agaricus bisporus            0.098361
Amanita phalloides           0.057377
Armillaria mellea            0.087432
Boletus edulis               0.087432
Cantharellus cibarius        0.035519
Clitocybe dealbata           0.073770
Entoloma sinuatum            0.120219
Hygrophoropsis aurantiaca    0.101093
Hypholoma fasciculare        0.076503
Lactarius resimus            0.068306
Pleurotus ostreatus          0.076503
Rubroboletus satanas         0.117486
Name: proportion, dtype: float64

In [8]:
# Convertit les DataFrames Pandas en tables PyArrow.
#
# preserve_index=False évite d'ajouter l'index Pandas
# comme nouvelle colonne dans le fichier Parquet.
train_table = pa.Table.from_pandas(
    train_df,
    preserve_index=False
)

validation_table = pa.Table.from_pandas(
    validation_df,
    preserve_index=False
)

test_table = pa.Table.from_pandas(
    test_df,
    preserve_index=False
)


In [9]:
print(train_df)


# Sauvegarde le dataset d'entraînement dans S3.
pq.write_table(
    train_table,
    TRAIN_PATH,
    filesystem=s3,
    compression="zstd"
)

# Sauvegarde le dataset de validation dans S3.
pq.write_table(
    validation_table,
    VALIDATION_PATH,
    filesystem=s3,
    compression="zstd"
)

# Sauvegarde le dataset de test dans S3.
pq.write_table(
    test_table,
    TEST_PATH,
    filesystem=s3,
    compression="zstd"
)


print("Les trois fichiers ont été sauvegardés dans S3.")

                                                  image  \
2208  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
1323  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
1374  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
2149  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
488   {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
...                                                 ...   
211   {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
1635  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
1291  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
931   {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
1123  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   

                                    label                            bbox  \
2208             Agaricus_bisporus_edible      [85.0, 67.0, 519.0, 529.5]   
1323      Hypholoma_fasciculare_poisonous       [0.0, 34.0, 625.0, 639.5]   
1374      Hypholoma_fasciculare_poisonous     [51.0, 321.0, 

Les trois fichiers ont été sauvegardés dans S3.
